# Vietnamese Medical Visual Question Answering (VQA)
## Modular Architecture with Cross-Modal Co-Attention and LLM-Enhanced Refinement

This notebook implements a comprehensive Medical VQA system for the Vietnamese language, utilizing a hybrid dataset derived from SLAKE and VQA-RAD. The pipeline incorporates state-of-the-art vision-language techniques, including PhoBERT for Vietnamese text encoding and DenseNet-121 (XRV) for medical image feature extraction.

### Research Methodology and Variants
The project evaluates four distinct architectural configurations:

| Variant | Visual Backbone | Textual Backbone | Decoder Strategy | Description |
|:---:|:---:|:---:|:---:|:---|
| **A1** | DenseNet-121 (XRV) | PhoBERT | LSTM + Attention | Baseline Modular Architecture |
| **A2** | DenseNet-121 (XRV) | PhoBERT | Transformer Decoder | Advanced Modular with Beam Search |
| **B1** | LLaVA-Med-7B | — | — | Zero-shot Foundation Model Assessment |
| **B2** | LLaVA-Med-7B | — | — | LoRA + DPO | Supervised Fine-Tuning with RLHF Alignment |

### Core Technologies
- **Fusion Mechanism:** Parallel Co-Attention (Kim et al., 2018).
- **Language Modeling:** PhoBERT (Nguyen & Nguyen, 2020).
- **Optimization:** Direct Preference Optimization (DPO) for clinical hallucination reduction.

## 0. Project Repository Initialization
Clone the project repository from GitHub and configure the environment for modular access.

In [ ]:
import os
import sys

# 1. CONFIGURATION: Using your official repository
REPO_URL = "https://github.com/QuangVoAI/DL_MedicalVQA_Project.git"
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

# 2. CLONING
if not os.path.exists(REPO_NAME):
    print(f"Cloning repository: {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Repository {REPO_NAME} already exists. Pulling latest changes...")
    !cd {REPO_NAME} && git pull

# 3. WORKSPACE SETUP
os.chdir(os.path.join(os.getcwd(), REPO_NAME))

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
    
print(f"Successfully initialized workspace at: {os.getcwd()}")

## 1. Environment Configuration and Workspace Setup
Installation of necessary dependencies and configuration of the Python path for modular script access.

In [ ]:
import torch
import yaml
import pandas as pd
from datasets import load_dataset

# Install dependencies (if needed on Kaggle)
# !pip install -r requirements.txt

# Hardware Acceleration Detection
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() 
    else 'mps' if torch.backends.mps.is_available() 
    else 'cpu'
)

print(f"Device: {DEVICE}")
print(f"PyTorch Version: {torch.__version__}")

## 2. Dataset Integration from Hugging Face Hub
Loading the refined Vietnamese Medical VQA dataset (Gold Standard) directly from the remote repository.

In [ ]:
REPO_ID = "SpringWang08/medical-vqa-vi"

print(f"Synchronizing dataset with remote repository: {REPO_ID}...")
hf_dataset = load_dataset(REPO_ID)

print("\nDataset Split Statistics:")
for split in hf_dataset.keys():
    print(f" - {split.capitalize()}: {len(hf_dataset[split])} samples")

## 3. Global Hyperparameter and System Configuration
Parsing the YAML configuration file to initialize model dimensions, training schedules, and evaluation parameters.

In [ ]:
with open('configs/medical_vqa.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print("System Parameters Initialized:")
print(f" - Image Size: {cfg['data']['image_size']}")
print(f" - Text Embedding: {cfg['model_a']['phobert_model']}")
print(f" - Batch Size: {cfg['train']['batch_size']}")

## 4. Data Pipeline and Tokenization
Implementation of custom PyTorch DataLoaders with real-time image normalization and PhoBERT tokenization.

In [ ]:
from transformers import AutoTokenizer
from src.data.medical_dataset import MedicalVQADataset
from torch.utils.data import DataLoader
from src.utils.visualization import MedicalImageTransform

tokenizer = AutoTokenizer.from_pretrained(cfg['model_a']['phobert_model'])
transform = MedicalImageTransform(size=cfg['data']['image_size'])

train_ds = MedicalVQADataset(hf_dataset=hf_dataset['train'], tokenizer=tokenizer, transform=transform)
val_ds = MedicalVQADataset(hf_dataset=hf_dataset['validation'], tokenizer=tokenizer, transform=transform)
test_ds = MedicalVQADataset(hf_dataset=hf_dataset['test'], tokenizer=tokenizer, transform=transform)

train_loader = DataLoader(train_ds, batch_size=cfg['train']['batch_size'], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg['train']['batch_size'], shuffle=False)
test_loader = DataLoader(test_ds, batch_size=cfg['train']['batch_size'], shuffle=False)

print(f"DataLoaders successfully initialized.")

## 5. Implementation of Variant A (Modular Architectures)
Construction of the vision-language fusion model using DenseNet-121 XRV and PhoBERT.

In [ ]:
from src.models.medical_vqa_model import MedicalVQAModelA

model_A1 = MedicalVQAModelA(decoder_type='lstm', vocab_size=30000).to(DEVICE)
model_A2 = MedicalVQAModelA(decoder_type='transformer', vocab_size=30000).to(DEVICE)

print(f"Variant A1 Parameters: {sum(p.numel() for p in model_A1.parameters())/1e6:.1f}M")
print(f"Variant A2 Parameters: {sum(p.numel() for p in model_A2.parameters())/1e6:.1f}M")

## 6. Model Training and Optimization
Execution of the training loop for modular variants. Cross-entropy loss with label smoothing is utilized.

In [ ]:
from src.engine.trainer import Trainer
import torch.nn as nn

criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
trainer = Trainer(
    model=model_A1, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    criterion=criterion, 
    device=DEVICE,
    checkpoint_dir='checkpoints/A1'
)

# trainer.train(epochs=cfg['train']['epochs']) # Uncomment to begin training

## 7. Comparative Performance Analysis
Quantitative evaluation across standard VQA and NLP metrics (BLEU, ROUGE-L, BERTScore).

In [ ]:
from src.engine.medical_eval import MedicalVQAEvaluator
evaluator = MedicalVQAEvaluator(device=DEVICE, tokenizer=tokenizer)
print("Evaluation engine ready.")

## 8. Clinical Conclusion and Scientific Documentation
**References:**
- PhoBERT: Nguyen & Nguyen, Findings of EMNLP 2020.
- DPO: Rafailov et al., NeurIPS 2023.
- LLaVA-Med: Li et al., arXiv 2023.